# Basis-independent cage manifold and chiral-index diagnostics

This notebook tests whether the full nine-dimensional square-QDM $(0,4)$ cage manifold survives after arbitrary basis rotation, then separates index-protected from paired chiral zero modes.

In [ ]:
from pathlib import Path
import sys
import numpy as np
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (CageSearchConfig, CageSearcher, diagnose_chiral_index,
    fixed_cage_manifold_compatibility_from_hamiltonians, partition_cage_hamiltonian)
from qlinks.models import SquareQDMModel

In [ ]:
model = SquareQDMModel(lx=4, ly=4, boundary_condition="periodic", winding_x=0, winding_y=0, winding_convention="electric", coup_kin=1.0, coup_pot=1.0)
build = model.build(basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True)
search = CageSearcher.from_model_build_result(build, config=CageSearchConfig(search_type="type1", tolerance=1e-10, degenerate_basis_strategy="ipr", ipr_n_restarts=16, ipr_candidate_count=16, ipr_random_seed=1234)).run()
records = tuple(search[(0, 4)])
full_states = []
for record in records:
    state = np.zeros(search.hilbert_size, dtype=np.complex128)
    state[np.asarray(record.cage_state.support, dtype=int)] = np.asarray(record.cage_state.local_state)
    full_states.append(state)
manifold_states = np.column_stack(full_states)
support = np.flatnonzero(np.max(np.abs(manifold_states), axis=1) > 1e-10)

In [ ]:
term_builder = SparseHamiltonianBuilder(backend="scipy", dtype=np.complex128, on_missing="raise")
local_terms = tuple(term_builder.build(build.basis, [op]) for op in build.kinetic_operators + build.potential_operators)
manifold_report = fixed_cage_manifold_compatibility_from_hamiltonians(build.hamiltonian, local_terms, support=support, manifold_states=manifold_states, tolerance=1e-10)
manifold_report.to_summary_dict()

In [ ]:
boundary = partition_cage_hamiltonian(build.kinetic, support).boundary
chiral_report = diagnose_chiral_index(boundary, tolerance=1e-10)
chiral_report.to_summary_dict()

## Interpretation

The full nine-dimensional manifold has only 11 exact local compatibility directions. Allowing arbitrary unitary rotation inside the manifold therefore does not recover the 44-dimensional robustness of the eight localized records.

For the active support-boundary block, $A$ has shape $64\times48$, rank 39, $\dim\ker A=9$, and $\dim\ker A^\dagger=25$. Hence $\mathrm{ind}(A)=9-25=-16$. The index protects 16 zero modes on the boundary sublattice, not the nine cage modes on the support sublattice. The cage modes are paired chiral zero modes and need extra interference structure for stability.